In [48]:
from dotenv import load_dotenv
import operator
import json
from uuid import UUID
from enum import Enum
from pydantic import BaseModel
from typing import Annotated, List, Literal, TypedDict
from langchain.chat_models import init_chat_model
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from openai import OpenAI
from groq import Groq


load_dotenv()
client = Groq()

# Example usage: Create a chat completion
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of low-latency LLMs.",
        }
    ],
    model="openai/gpt-oss-20b",
)

print(chat_completion.choices[0].message.content)

### Why Low‑Latency Large Language Models Matter

| Domain | Why latency is critical | Typical latency budget |
|--------|------------------------|------------------------|
| **Conversational agents** (chatbots, virtual assistants) | Users expect a *fluid* conversation; a pause longer than a few hundred milliseconds feels like a “thinking” delay. | < 200 ms (per response) |
| **Real‑time translation & captioning** | Live events, broadcasts, or video‑chat require instant subtitles or translations. | < 1 s |
| **Interactive coding assistants** | Developers test code snippets on the fly; a 1‑second delay can interrupt workflow. | < 500 ms |
| **Augmented/Virtual Reality (AR/VR)** | Latency > 20 ms can cause motion sickness and break immersion. | < 20 ms |
| **Autonomous systems** (driving, robotics) | Decisions must be taken in real time to avoid accidents. | < 50 ms |
| **Edge / IoT devices** | No network round‑trip; the model must finish on‑device to keep the device responsive. | < 50 m

In [49]:
class ObservationType(Enum):
    'skill'
    'deficit'

In [50]:
class Skill(TypedDict):
    id: UUID
    name: str
    parent_id: UUID
    status: str
    confidence: float
    observations: list[UUID]


In [51]:
class LearnerObservation(TypedDict):
    id: UUID
    type: ObservationType
    text: str

In [52]:
class CoachState(TypedDict):
    query:str
    conversation:list[str]


In [53]:
def load_prompt(prompt_file:str):
    prompt = ''
    with open(prompt_file, 'r', encoding='utf-8') as file:
        data = json.load(file)
        for section in ['role','task','query','context','output','format','constraints']:
            if data[section] and data[section] != '':
                prompt += f"{data[section]}\n"

    return prompt

In [60]:
def load_query(query_file:str):
    with open(query_file, 'r', encoding='utf-8') as file:
        data = json.load(file)
        if data['query'] and data['query'] != '':
            return data['query']
        else:
            return None

In [61]:
load_prompt('EVAL_PROMPT_v1-0.json')
load_query('query_1.json')

'I have the initial value problem dy/dx = -6xy, y(0)=7. How do I find the integral of -6xy?'

In [56]:
eval_model = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq",
    temperature=0
)

In [64]:
def analyze_prompt(state: CoachState, prompt:str) -> Literal["subject_list"]:
    query = state['query']
    context = state['conversation']

    prompt = prompt.format(query=query, context=context)
    response = eval_model.invoke([{"role":"user","content":prompt}])
    print(response)

In [71]:
prompt = load_prompt('STRATEGY_PROMPT_v1-2.json')
query = load_query('query_3.json')
state = CoachState(query=query, conversation=[])
analyze_prompt(state, prompt)

content='The user is confusing algebraic multiplicity with the number of linearly independent eigenvectors, not realizing that a defective matrix can have fewer eigenvectors than the multiplicity of its eigenvalue. Confidence:0.85' additional_kwargs={'reasoning_content': 'We need to produce two hypotheses about the primary knowledge or skill gap causing the user to ask this query. The user asks: "If the algebraic multiplicity tells us how many times an eigenvalue appears as a root, why doesn\'t the geometric multiplicity always equal it? Where do the \'missing\' eigenvectors go?" So they are confused about the difference between algebraic multiplicity and geometric multiplicity. They might think that eigenvectors correspond to each root, but missing eigenvectors are due to defective matrices. They might not understand the concept of generalized eigenvectors or Jordan chains. They might think that eigenvectors are missing because they are not counted, but they actually exist as generali